# Orbit Wars: Borg Agency Collective Planner v5

This notebook builds a clean, reproducible Orbit Wars submission archive for the **Borg Agency Collective Planner**. It overwrites the original ProducerLite planner behavior with Borg-style collective target assimilation, multi-fraction safe-drain waves, leader pressure, defensive preservation, regrouping, and deterministic validation.

It writes `/kaggle/working/submission.tar.gz`, validates the archive, compiles `main.py`, and stops there.

**No automatic submission is performed.** Submit the generated archive from the Kaggle UI when ready.


## Agent Notes

- Base package retained: `orbit_lite` movement, intercept, garrison-flow, and action adapters.
- Planner overwritten: `plan_lite_waves` now runs the Borg Agency collective planner.
- Borg behavior: multi-fraction assimilation waves, production capture, neutral absorption, 4P leader pressure, endangered-owned-world defense, and pressure-gradient regroup.
- Determinism: stable candidate ranking, exact sparse flow scoring, SHA-256 archive validation, and Python compile validation.
- Output: `/kaggle/working/submission.tar.gz`.


In [1]:
from pathlib import Path
import base64
import gzip
import hashlib
import shutil
import tarfile

WORK_DIR = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_PATH = WORK_DIR / 'submission.tar.gz'
SRC_DIR = WORK_DIR / 'submission_src'

print('Output archive:', ARCHIVE_PATH)
print('Validation source dir:', SRC_DIR)


Output archive: /kaggle/working/submission.tar.gz
Validation source dir: /kaggle/working/submission_src


In [2]:
# Hidden payload: Borg Agency compressed tar containing main.py and the orbit_lite package.
EMBEDDED_ARCHIVE_SHA256 = '87dfb935e41e99ccfd96860eb61e627e8a7d4066252c034f00d73f092b80bc75'
EMBEDDED_MAIN_SHA256 = '5e0cb80acc0ec4087bc5b88d5036446fcc8efaaa56a971547fa61f9084d1c541'
EMBEDDED_ARCHIVE_B64_GZ = '''
...
/v3277d//3H//b+6pu3EAHADAA==
'''



In [3]:
archive_payload = base64.b64decode(EMBEDDED_ARCHIVE_B64_GZ)
archive_sha = hashlib.sha256(archive_payload).hexdigest()
if archive_sha != EMBEDDED_ARCHIVE_SHA256:
    raise RuntimeError(f'archive sha mismatch: {archive_sha} != {EMBEDDED_ARCHIVE_SHA256}')

ARCHIVE_PATH.write_bytes(archive_payload)
print(f'wrote {ARCHIVE_PATH}')
print(f'submission.tar.gz size: {ARCHIVE_PATH.stat().st_size:,} bytes')
print(f'submission.tar.gz sha256: {archive_sha}')

if SRC_DIR.exists():
    shutil.rmtree(SRC_DIR)
SRC_DIR.mkdir(parents=True, exist_ok=True)
with tarfile.open(ARCHIVE_PATH, 'r:gz') as tar:
    tar.extractall(SRC_DIR)

main_path = SRC_DIR / 'main.py'
main_sha = hashlib.sha256(main_path.read_bytes()).hexdigest()
print(f'main.py sha256: {main_sha}')
if main_sha != EMBEDDED_MAIN_SHA256:
    raise RuntimeError(f'main.py sha mismatch: {main_sha} != {EMBEDDED_MAIN_SHA256}')


wrote /kaggle/working/submission.tar.gz
submission.tar.gz size: 54,080 bytes
submission.tar.gz sha256: 1d0d2749846659c353b6231c4ea95bc4c44a78dc6bce0aee7871b0463d052f70
main.py sha256: 26da0a48e25bcc69f1c680186916330643d338181bc29a2db8b52b1f461826e3


/tmp/ipykernel_16/3589897195.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(SRC_DIR)


In [4]:
import importlib.util
import py_compile
import sys

main_path = SRC_DIR / 'main.py'
py_compile.compile(str(main_path), doraise=True)

sys.path.insert(0, str(SRC_DIR))
spec = importlib.util.spec_from_file_location('producer_hybrid_v4', main_path)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)

assert callable(getattr(module, 'agent', None)), 'main.py does not expose callable agent(obs)'
assert (SRC_DIR / 'orbit_lite').exists(), 'orbit_lite package missing from archive'
print('Validation passed: archive extracts, main.py compiles, and agent(obs) is callable.')


Validation passed: archive extracts, main.py compiles, and agent(obs) is callable.


In [5]:
RUN_LOCAL_MATCH = False

if RUN_LOCAL_MATCH:
    try:
        from kaggle_environments import make
    except Exception:
        import subprocess
        import sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-qU', 'kaggle-environments'])
        from kaggle_environments import make

    env = make('orbit_wars', debug=True)
    env.run([str(SRC_DIR / 'main.py'), 'random'])
    for idx, state in enumerate(env.steps[-1]):
        print(f'player={idx} status={state.status} reward={state.reward}')
else:
    print('Optional match skipped. Set RUN_LOCAL_MATCH=True for a slower environment smoke match.')


Optional match skipped. Set RUN_LOCAL_MATCH=True for a slower environment smoke match.


In [6]:
from IPython.display import Markdown

Markdown(f'''
### Ready

Use `{ARCHIVE_PATH}` as the Kaggle submission artifact.

This notebook intentionally does not submit automatically.
''')



### Ready

Use `/kaggle/working/submission.tar.gz` as the Kaggle submission artifact.

This notebook intentionally does not submit automatically.
